# Western Armenian Corpus — MongoDB EDA

Quick exploratory analysis of the `western_armenian_corpus` database.  
Covers: document counts, word counts, source coverage, language distribution, data quality spot-checks.

In [ ]:
import pandas as pd
from pymongo import MongoClient

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

# Format integer columns with commas
def fmt_int_cols(df):
    """Apply comma formatting to integer-like columns in a DataFrame."""
    df = df.copy()
    for col in df.columns:
        if df[col].dtype in ("int64", "int32", "float64"):
            # Check if all non-null values are whole numbers
            vals = df[col].dropna()
            if len(vals) > 0 and (vals == vals.astype(int)).all():
                df[col] = vals.astype(int).map("{:,}".format)
    return df

client = MongoClient("mongodb://localhost:27017/")
db = client["western_armenian_corpus"]
docs = db["documents"]

total = docs.estimated_document_count()
print(f"Total documents in collection: {total:,}")

Total documents in collection: 1,016,477


## 1. Documents by Source

File count, total word count, and average word count per source.

In [ ]:
pipeline = [
    {"$group": {
        "_id": "$source",
        "doc_count": {"$sum": 1},
        "total_words": {"$sum": {"$ifNull": ["$metadata.word_count", 0]}},
        "total_chars": {"$sum": {"$ifNull": ["$metadata.char_count", 0]}},
        "avg_words": {"$avg": {"$ifNull": ["$metadata.word_count", 0]}},
        "min_words": {"$min": "$metadata.word_count"},
        "max_words": {"$max": "$metadata.word_count"},
    }},
    {"$sort": {"total_words": -1}},
]
rows = list(docs.aggregate(pipeline))
df_source = pd.DataFrame(rows).rename(columns={"_id": "source"})
df_source["avg_words"] = df_source["avg_words"].round(1)

# Grand totals row
totals = pd.DataFrame([{
    "source": "** TOTAL **",
    "doc_count": df_source["doc_count"].sum(),
    "total_words": df_source["total_words"].sum(),
    "total_chars": df_source["total_chars"].sum(),
    "avg_words": df_source["total_words"].sum() / max(df_source["doc_count"].sum(), 1),
    "min_words": df_source["min_words"].min(),
    "max_words": df_source["max_words"].max(),
}])
display(fmt_int_cols(pd.concat([df_source, totals], ignore_index=True)))

,source,doc_count,total_words,total_chars,avg_words,min_words,max_words
0,archive_org,"1,570","162,692,082","962,811,438","103,625.50",53,"1,818,103"
1,opus_nllb_hy,"602,444","120,650,162","851,148,572",200.30,80,568
2,wikipedia_ea,"330,346","108,769,639","833,771,168",329.30,1,"21,651"
3,opus_multiccaligned_hy,"34,933","11,932,380","90,750,888",341.60,98,873
4,opus_ccaligned_hy,"24,814","10,861,594","82,816,483",437.70,165,978
5,wikipedia,"13,619","4,745,500","38,206,405",348.40,4,"19,565"
6,hamazkayin:pakine,965,"940,921","6,540,188",975.00,33,"10,106"
7,opus_wikimedia_hy,866,"816,209","6,263,239",942.50,153,"4,774"
8,opus_ted2020_hy,"1,663","496,116","3,369,777",298.30,143,715
9,opus_qed_hy,"1,546","456,508","3,038,379",295.30,34,"1,650"


## 2. Documents by Language Code

Dialect distribution (hyw = Western Armenian, hye = Eastern Armenian).

In [ ]:
pipeline_lang = [
    {"$group": {
        "_id": {"$ifNull": ["$metadata.internal_language_branch", "unknown"]},
        "doc_count": {"$sum": 1},
        "total_words": {"$sum": {"$ifNull": ["$metadata.word_count", 0]}},
        "avg_words": {"$avg": {"$ifNull": ["$metadata.word_count", 0]}},
    }},
    {"$sort": {"doc_count": -1}},
]
df_lang = pd.DataFrame(list(docs.aggregate(pipeline_lang))).rename(columns={"_id": "language_code"})
df_lang["avg_words"] = df_lang["avg_words"].round(1)
display(fmt_int_cols(df_lang))

,language_code,doc_count,total_words,avg_words
0,hyw,"682,721","253,978,484",372.00
1,hye,"331,723","169,057,847",509.60
2,hy,"1,327","21,245",16.00
3,eng,635,"85,661",134.90
4,und,71,"1,414",19.90


## 3. Source × Language Code Cross-Tab

Which sources contribute WA vs EA content?

In [23]:
pipeline_cross = [
    {"$group": {
        "_id": {
            "source": "$source",
            "lang": {"$ifNull": ["$metadata.language_code", "unknown"]},
        },
        "doc_count": {"$sum": 1},
        "total_words": {"$sum": {"$ifNull": ["$metadata.word_count", 0]}},
    }},
    {"$sort": {"_id.source": 1}},
]
rows_cross = list(docs.aggregate(pipeline_cross))
flat = [{"source": r["_id"]["source"], "language_code": r["_id"]["lang"],
         "doc_count": r["doc_count"], "total_words": r["total_words"]} for r in rows_cross]
df_cross = pd.DataFrame(flat)

# Pivot: doc count
pivot_docs = df_cross.pivot_table(index="source", columns="language_code",
                                   values="doc_count", aggfunc="sum", fill_value=0, margins=True)
print("=== Document Count by Source × Language Code ===")
display(fmt_int_cols(pivot_docs))

# Pivot: word count
pivot_words = df_cross.pivot_table(index="source", columns="language_code",
                                    values="total_words", aggfunc="sum", fill_value=0, margins=True)
print("\n=== Word Count by Source × Language Code ===")
display(fmt_int_cols(pivot_words))

=== Document Count by Source × Language Code ===


language_code,eng,hy,hye,hyw,und,All
source,,,,,,
archive_org,0,0,637,933,0,"1,570"
dpla,417,"1,327",0,0,71,"1,815"
eastern_armenian_news:a1plus,0,0,162,0,0,162
eastern_armenian_news:aravot,0,0,135,0,0,135
eastern_armenian_news:armenpress,0,0,67,0,0,67
eastern_armenian_news:armtimes,0,0,163,0,0,163
hamazkayin:main,0,0,10,152,0,162
hamazkayin:pakine,0,0,4,961,0,965
opus_bible_hy,0,0,0,646,0,646



=== Word Count by Source × Language Code ===


language_code,eng,hy,hye,hyw,und,All
source,,,,,,
archive_org,0,0,"59,964,169","102,727,913",0,"162,692,082"
dpla,"19,154","21,245",0,0,"1,414","41,813"
eastern_armenian_news:a1plus,0,0,"59,069",0,0,"59,069"
eastern_armenian_news:aravot,0,0,"65,578",0,0,"65,578"
eastern_armenian_news:armenpress,0,0,"27,549",0,0,"27,549"
eastern_armenian_news:armtimes,0,0,"108,483",0,0,"108,483"
hamazkayin:main,0,0,"3,845","89,412",0,"93,257"
hamazkayin:pakine,0,0,"1,627","939,294",0,"940,921"
opus_bible_hy,0,0,0,"228,273",0,"228,273"


## 4. Documents by Source Type

Grouping by `metadata.source_type` (e.g. newspaper, literary, religious, parallel_corpus, etc.).

In [5]:
pipeline_stype = [
    {"$group": {
        "_id": {"$ifNull": ["$metadata.source_type", "not_set"]},
        "doc_count": {"$sum": 1},
        "total_words": {"$sum": {"$ifNull": ["$metadata.word_count", 0]}},
        "avg_words": {"$avg": {"$ifNull": ["$metadata.word_count", 0]}},
    }},
    {"$sort": {"total_words": -1}},
]
df_stype = pd.DataFrame(list(docs.aggregate(pipeline_stype))).rename(columns={"_id": "source_type"})
df_stype["avg_words"] = df_stype["avg_words"].round(1)
display(fmt_int_cols(df_stype))

,source_type,doc_count,total_words,avg_words
0,book,"1,570","162,692,082","103,625.50"
1,encyclopedia,"343,965","113,515,139",330.00
2,news,476,"242,337",509.10
3,library,"1,815","41,813",23.00


## 5. Processing Status

How many documents have been normalized, deduplicated, filtered, dialect-classified?

In [6]:
processing_flags = ["normalized", "deduplicated", "filtered", "dialect_classified"]
proc_rows = []
for flag in processing_flags:
    true_count = docs.count_documents({f"processing.{flag}": True})
    false_count = docs.count_documents({f"processing.{flag}": {"$ne": True}})
    proc_rows.append({"flag": flag, "true": true_count, "false_or_missing": false_count,
                       "pct_done": f"{100 * true_count / max(true_count + false_count, 1):.1f}%"})

display(fmt_int_cols(pd.DataFrame(proc_rows)))

,flag,true,false_or_missing,pct_done
0,normalized,0,"347,826",0.0%
1,deduplicated,0,"347,826",0.0%
2,filtered,0,"347,826",0.0%
3,dialect_classified,0,"347,826",0.0%


## 6. Missing Metadata Audit

Check how many documents are missing key metadata fields.

In [24]:
fields_to_check = [
    "source", "title", "text",
    "metadata.url", "metadata.word_count", "metadata.char_count",
    "metadata.language_code", "metadata.dialect", "metadata.source_type",
    "metadata.author", "metadata.wa_score", "metadata.date_scraped",
    "metadata.document_metrics", "content_hash",
]

audit_rows = []
for field in fields_to_check:
    # Single query: field missing OR null OR empty string
    total_missing = docs.count_documents({"$or": [
        {field: {"$exists": False}},
        {field: None},
        {field: ""},
    ]})
    present = total - total_missing
    audit_rows.append({
        "field": field,
        "present": f"{present:,}",
        "missing_or_empty": f"{total_missing:,}",
        "pct_missing": f"{100 * total_missing / max(total, 1):.1f}%",
    })

display(pd.DataFrame(audit_rows))

,field,present,missing_or_empty,pct_missing
0,source,"1,016,477",0,0.0%
1,title,"1,016,477",0,0.0%
2,text,"1,016,476",1,0.0%
3,metadata.url,"5,327","1,011,150",99.5%
4,metadata.word_count,"1,016,477",0,0.0%
5,metadata.char_count,"1,016,477",0,0.0%
6,metadata.language_code,"1,016,477",0,0.0%
7,metadata.dialect,"668,755","347,722",34.2%
8,metadata.source_type,"1,016,477",0,0.0%
9,metadata.author,"1,413","1,015,064",99.9%


## 7. Word Count Distribution

Histogram buckets to see how document lengths are distributed.

In [25]:
pipeline_wc = [
    {"$match": {"metadata.word_count": {"$exists": True, "$ne": None}}},
    {"$bucket": {
        "groupBy": "$metadata.word_count",
        "boundaries": [0, 50, 100, 250, 500, 1000, 2500, 5000, 10000, 50000],
        "default": "50K+",
        "output": {"count": {"$sum": 1}, "total_words": {"$sum": "$metadata.word_count"}},
    }},
]
rows_wc = list(docs.aggregate(pipeline_wc))
df_wc = pd.DataFrame(rows_wc).rename(columns={"_id": "bucket"})
df_wc["bucket"] = df_wc["bucket"].astype(str)
display(fmt_int_cols(df_wc))

,bucket,count,total_words
0,0,"45,317","1,268,599"
1,50,"39,711","2,853,198"
2,100,"688,234","129,328,625"
3,250,"181,410","59,087,709"
4,500,"39,233","25,855,504"
5,1000,"17,126","24,994,597"
6,2500,"3,039","10,316,852"
7,5000,966,"6,472,207"
8,10000,605,"14,736,194"
9,50K+,836,"148,231,166"


## 8. WA Score Distribution (per source)

For sources that have `metadata.wa_score`, show min/avg/max to see dialect purity.

In [26]:
pipeline_wa = [
    {"$match": {"metadata.wa_score": {"$exists": True, "$ne": None}}},
    {"$group": {
        "_id": "$source",
        "doc_count": {"$sum": 1},
        "min_wa": {"$min": "$metadata.wa_score"},
        "avg_wa": {"$avg": "$metadata.wa_score"},
        "max_wa": {"$max": "$metadata.wa_score"},
        "median_sample": {"$push": "$metadata.wa_score"},
    }},
    {"$sort": {"avg_wa": -1}},
]
rows_wa = list(docs.aggregate(pipeline_wa))
for r in rows_wa:
    scores = sorted(r.pop("median_sample", []))
    r["p50_wa"] = scores[len(scores) // 2] if scores else None

df_wa = pd.DataFrame(rows_wa).rename(columns={"_id": "source"})
for col in ["min_wa", "avg_wa", "max_wa", "p50_wa"]:
    df_wa[col] = df_wa[col].round(2)
df_wa["doc_count"] = df_wa["doc_count"].map("{:,}".format)
display(df_wa)

,source,doc_count,min_wa,avg_wa,max_wa,p50_wa
0,opus_wikimedia_hyw,14,126.00,182.00,237.00,186.00
1,hamazkayin:main,162,0.00,154.82,274.50,183.50
2,hamazkayin:pakine,965,1.00,129.33,316.50,90.00
3,opus_bible_hy,646,36.00,103.18,188.00,101.50
4,opus_wikimedia_hy,866,0.00,41.23,116.00,41.00
5,opus_ccaligned_hy,"24,814",8.50,40.47,140.50,39.00
6,opus_nllb_hy,"602,444",-4.00,35.87,136.50,34.50
7,opus_multiccaligned_hy,"34,933",-0.50,35.82,135.00,35.00
8,opus_ted2020_hy,"1,663",-5.50,35.59,79.00,35.00
9,opus_qed_hy,"1,546",-0.50,34.87,104.00,34.50


## 9. Scrape Timeline

When were documents ingested? Shows scrape activity over time.

In [27]:
pipeline_timeline = [
    {"$match": {"metadata.date_scraped": {"$exists": True, "$ne": None}}},
    {"$group": {
        "_id": {
            "date": {"$dateToString": {"format": "%Y-%m-%d", "date": "$metadata.date_scraped"}},
            "source": "$source",
        },
        "doc_count": {"$sum": 1},
    }},
    {"$sort": {"_id.date": -1}},
]
rows_tl = list(docs.aggregate(pipeline_timeline))
flat_tl = [{"date": r["_id"]["date"], "source": r["_id"]["source"],
            "doc_count": r["doc_count"]} for r in rows_tl]
df_timeline = pd.DataFrame(flat_tl)

if not df_timeline.empty:
    # Summary by date
    by_date = df_timeline.groupby("date")["doc_count"].sum().reset_index().sort_values("date", ascending=False)
    print("=== Documents ingested per day (most recent first) ===")
    display(by_date.head(30))

    # Summary by date × source (last 10 days)
    recent_dates = by_date["date"].head(10).tolist()
    recent = df_timeline[df_timeline["date"].isin(recent_dates)]
    pivot_tl = recent.pivot_table(index="date", columns="source", values="doc_count",
                                   aggfunc="sum", fill_value=0).sort_index(ascending=False)
    print("\n=== Recent ingestion by source (last 10 active days) ===")
    display(pivot_tl)
else:
    print("No date_scraped data found.")

=== Documents ingested per day (most recent first) ===


,date,doc_count
2,2026-03-14,668651
1,2026-03-13,2291
0,2026-03-12,345535



=== Recent ingestion by source (last 10 active days) ===


source,archive_org,dpla,eastern_armenian_news:a1plus,eastern_armenian_news:aravot,eastern_armenian_news:armenpress,eastern_armenian_news:armtimes,hamazkayin:main,hamazkayin:pakine,opus_bible_hy,opus_ccaligned_hy,...,rss_news:Aztag,rss_news:Civilnet,rss_news:EVN Report,rss_news:Google News - Armenia,rss_news:Hetq,rss_news:Horizon Weekly,rss_news:Massis Post,rss_news:OC Media,wikipedia,wikipedia_ea
date,,,,,,,,,,,,,,,,,,,,,
2026-03-14,0,0,41,10,0,0,162,965,646,24814,...,20,10,44,48,50,5,25,16,0,0
2026-03-13,0,1815,121,125,67,163,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2026-03-12,1570,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,13619,330346


## 10. Duplicate Detection

Check for duplicate content_hash values and near-duplicate titles within the same source.

In [28]:
# Exact duplicates (same content_hash)
pipeline_dup = [
    {"$group": {"_id": "$content_hash", "count": {"$sum": 1}, "sources": {"$addToSet": "$source"}}},
    {"$match": {"count": {"$gt": 1}}},
    {"$sort": {"count": -1}},
    {"$limit": 20},
]
dup_rows = list(docs.aggregate(pipeline_dup))
if dup_rows:
    df_dup = pd.DataFrame(dup_rows).rename(columns={"_id": "content_hash"})
    print(f"Found {len(dup_rows)} duplicate content_hash groups (showing top 20):")
    display(df_dup)
else:
    print("No exact content_hash duplicates found (unique index is working).")

# Cross-source duplicates: same normalized_content_hash, different source
pipeline_cross_dup = [
    {"$group": {
        "_id": "$normalized_content_hash",
        "count": {"$sum": 1},
        "sources": {"$addToSet": "$source"},
        "titles": {"$addToSet": "$title"},
    }},
    {"$match": {"count": {"$gt": 1}, "sources.1": {"$exists": True}}},
    {"$sort": {"count": -1}},
    {"$limit": 15},
]
cross_dup_rows = list(docs.aggregate(pipeline_cross_dup))
if cross_dup_rows:
    print(f"\nCross-source near-duplicates (same normalized text, different sources): {len(cross_dup_rows)}")
    for r in cross_dup_rows[:10]:
        titles = r["titles"][:3]
        print(f"  {r['count']}x across {r['sources']}: {titles}")
else:
    print("\nNo cross-source near-duplicates found.")

No exact content_hash duplicates found (unique index is working).

No cross-source near-duplicates found.


## 11. Spot Check — Sample Documents

Random sample of documents to visually inspect data quality. Shows first 200 chars of text.

In [29]:
sample = list(docs.aggregate([{"$sample": {"size": 15}}]))
spot_rows = []
for doc in sample:
    meta = doc.get("metadata") or {}
    text = doc.get("text", "") or ""
    spot_rows.append({
        "source": doc.get("source", "?"),
        "title": (doc.get("title", "") or "")[:60],
        "lang": meta.get("language_code", "?"),
        "words": meta.get("word_count", "?"),
        "wa_score": meta.get("wa_score"),
        "text_preview": text[:200].replace("\n", " "),
    })

df_spot = pd.DataFrame(spot_rows)
display(df_spot)

,source,title,lang,words,wa_score,text_preview
0,opus_nllb_hy,NLLB_hy chunk 337863,hyw,185,30.00,Մինչեւ 2 շաբաթը կարող է վիժում առաջանալ: Իրաքի իշխանությունը անում է այն ինչ...
1,wikipedia_ea,ՎԶԵԲ,hye,6,NaN,ՎԵՐԱՀՂՈՒՄ Վերակառուցման և զարգացման եվրոպական բանկ
2,wikipedia_ea,Վերեսի,hye,163,NaN,|ենթարկում = |զինանշան = |դրոշ = |lat_deg = 50 |lat_min = 20 |lat_sec...
3,opus_nllb_hy,NLLB_hy chunk 9223,hyw,241,23.00,"Դուք իհարկե ազատ եք փոխել դա, եթե այդքան հակված եք. Ինձ նման երեխաները թողնո..."
4,opus_nllb_hy,NLLB_hy chunk 137619,hyw,340,28.00,"Ինձ հետ նույն բանն է պատահում, ես չգիտեմ ինչ անել: Քո ցավի անկեղծությունը ին..."
5,wikipedia_ea,Սեպառա,hye,153,NaN,|ենթարկում = |պատկեր = |զինանշան = |դրոշ = |lat_deg = 58|lat_min = ...
6,wikipedia_ea,Եռավեռե,hye,154,NaN,|ենթարկում = |պատկեր = |զինանշան = |դրոշ = |lat_deg = 58|lat_min = ...
7,wikipedia_ea,Հյուդասպի ճակատամարտ,hye,436,NaN,"Հյուդասպի ճակատամարտը տեղի է ունեցել մ. թ. ա. 326 թ. հունիսին, Փենջաբում, Հյ..."
8,wikipedia_ea,Սերգեյ Օլդենբուրգ,hye,461,NaN,thumb|Օլդենբուրգը 1930 թվականին Սերգեյ Ֆեոդորովիչ Օլդենբուրգ (; 1863 թվական...
9,wikipedia_ea,Կազիմեժ Գուրսկի,hye,575,NaN,ՌԿՍ (Լվով)|? (?) |1940-1941| Սպարտակ (Լվով)|? (?) |1944| Դինամո (Լվով)|? (?)...


## 12. Short / Empty Document Audit

Documents with very low word count — may indicate scraper issues or boilerplate-only pages.

In [30]:
pipeline_short = [
    {"$match": {"$or": [
        {"metadata.word_count": {"$lt": 30}},
        {"metadata.word_count": {"$exists": False}},
        {"text": {"$in": [None, ""]}},
    ]}},
    {"$group": {
        "_id": "$source",
        "count": {"$sum": 1},
        "examples": {"$push": {"title": "$title", "wc": "$metadata.word_count", "url": "$metadata.url"}},
    }},
    {"$sort": {"count": -1}},
]
short_rows = list(docs.aggregate(pipeline_short))
if short_rows:
    print(f"Sources with very short (<30 words) or empty documents:")
    for r in short_rows:
        examples = r["examples"][:3]
        print(f"\n  {r['_id']}: {r['count']} docs")
        for ex in examples:
            print(f"    - \"{(ex.get('title') or '?')[:50]}\"  wc={ex.get('wc')}  url={str(ex.get('url',''))[:60]}")
else:
    print("No short/empty documents found.")

Sources with very short (<30 words) or empty documents:

  wikipedia_ea: 22061 docs
    - "Թարգմանական գրականություն"  wc=6  url=None
    - "Միջազգային կազմակերպությունների ցանկ"  wc=28  url=None
    - "Հայոց պատմության ժամանակագրություն"  wc=20  url=None

  dpla: 1437 docs
    - "Hayapatum : Patmichʻkʻ ew Patmutʻiwnk Hayotsʻ"  wc=0  url=http://catalog.hathitrust.org/Record/000244167
    - "Ayn sew ōrerun : (patkerner)"  wc=23  url=http://catalog.hathitrust.org/Record/008400591
    - "Haykakan nor matenagitutʻiwn ew hanragitaran hay k"  wc=17  url=http://catalog.hathitrust.org/Record/002705269

  wikipedia: 520 docs
    - "Ախպար (այլ կիրառումներ)"  wc=25  url=None
    - "14 Օգոստոս"  wc=14  url=None
    - "15 Դեկտեմբեր"  wc=29  url=None

  rss_news:Google News - Armenia: 48 docs
    - "Armenian Advocate: Why the Iran-US War Could Be a "  wc=16  url=https://news.google.com/rss/articles/CBMiekFVX3lxTFBDTFg5dmp
    - "Armenia reveals its artist and song for Vienna - e"  wc=9  url=https://

## 13. Collections Overview

List all collections in the database and their document counts.

In [31]:
coll_rows = []
for name in sorted(db.list_collection_names()):
    count = db[name].estimated_document_count()
    coll_rows.append({"collection": name, "documents": count})

df_colls = pd.DataFrame(coll_rows)
display(df_colls)
print(f"\nTotal collections: {len(df_colls)}")

,collection,documents
0,acquisition_priorities,0
1,augmentation_checkpoint,0
2,author_bibliography,0
3,author_chronology,0
4,author_profiles,0
5,author_research_summary,0
6,author_timeline,0
7,book_inventory,0
8,catalogs,3790
9,coverage_gaps,0



Total collections: 14


## 14. Language Code Distribution per Source

Shows `metadata.language_code` values per source — confirms WA/EA tagging is consistent.
`metadata.dialect` has been decomissioned in favour of `metadata.language_code`.

In [32]:
pipeline_lang_src = [
    {"$group": {
        "_id": {
            "source": "$source",
            "lang": {"$ifNull": ["$metadata.language_code", "not_set"]},
        },
        "doc_count": {"$sum": 1},
        "total_words": {"$sum": {"$ifNull": ["$metadata.word_count", 0]}},
    }},
    {"$sort": {"_id.source": 1}},
]
rows_ls = list(docs.aggregate(pipeline_lang_src))
flat_ls = [{"source": r["_id"]["source"], "language_code": r["_id"]["lang"],
            "doc_count": r["doc_count"], "total_words": r["total_words"]} for r in rows_ls]
df_lang_src = pd.DataFrame(flat_ls)
pivot_lang_src = df_lang_src.pivot_table(index="source", columns="language_code",
                                          values="doc_count", aggfunc="sum", fill_value=0, margins=True)
display(pivot_lang_src)

language_code,eng,hy,hye,hyw,und,All
source,,,,,,
archive_org,0,0,637,933,0,1570
dpla,417,1327,0,0,71,1815
eastern_armenian_news:a1plus,0,0,162,0,0,162
eastern_armenian_news:aravot,0,0,135,0,0,135
eastern_armenian_news:armenpress,0,0,67,0,0,67
eastern_armenian_news:armtimes,0,0,163,0,0,163
hamazkayin:main,0,0,10,152,0,162
hamazkayin:pakine,0,0,4,961,0,965
opus_bible_hy,0,0,0,646,0,646


## 15. Distinct Values for Key Fields

Quick view of what unique values exist for source, language_code, dialect, source_type.

In [33]:
for field_label, field_path in [
    ("source", "source"),
    ("language_code", "metadata.language_code"),
    ("source_type", "metadata.source_type"),
    ("content_type", "metadata.content_type"),
    ("writing_category", "metadata.writing_category"),
]:
    values = docs.distinct(field_path)
    print(f"{field_label} ({len(values)} distinct): {sorted(str(v) for v in values if v)}")

source (34 distinct): ['archive_org', 'dpla', 'eastern_armenian_news:a1plus', 'eastern_armenian_news:aravot', 'eastern_armenian_news:armenpress', 'eastern_armenian_news:armtimes', 'hamazkayin:main', 'hamazkayin:pakine', 'opus_bible_hy', 'opus_ccaligned_hy', 'opus_gnome_hy', 'opus_kde4_hy', 'opus_multiccaligned_hy', 'opus_nllb_hy', 'opus_opensubtitles_hy', 'opus_qed_hy', 'opus_tatoeba_hy', 'opus_ted2020_hy', 'opus_ubuntu_hy', 'opus_wikimedia_hy', 'opus_wikimedia_hyw', 'rss_news:Agos', 'rss_news:Armenian Mirror-Spectator', 'rss_news:Armenian Weekly', 'rss_news:Aztag', 'rss_news:Civilnet', 'rss_news:EVN Report', 'rss_news:Google News - Armenia', 'rss_news:Hetq', 'rss_news:Horizon Weekly', 'rss_news:Massis Post', 'rss_news:OC Media', 'wikipedia', 'wikipedia_ea']
language_code (5 distinct): ['eng', 'hy', 'hye', 'hyw', 'und']
source_type (7 distinct): ['book', 'cultural', 'dataset', 'encyclopedia', 'library', 'literary', 'news']
content_type (3 distinct): ['article', 'literature', 'parallel_

## 16. Top Titles by Word Count

Largest individual documents — useful for spotting bulk imports, OCR dumps, or concatenated texts.

In [34]:
top_docs = list(docs.find(
    {"metadata.word_count": {"$exists": True}},
    {"source": 1, "title": 1, "metadata.word_count": 1, "metadata.language_code": 1, "metadata.url": 1},
).sort("metadata.word_count", -1).limit(25))

top_rows = [{
    "source": d.get("source"),
    "title": (d.get("title") or "")[:60],
    "word_count": (d.get("metadata") or {}).get("word_count"),
    "lang": (d.get("metadata") or {}).get("language_code", "?"),
    "url": str((d.get("metadata") or {}).get("url", ""))[:60],
} for d in top_docs]
display(pd.DataFrame(top_rows))

,source,title,word_count,lang,url
0,archive_org,中国五音十二律论文 / Chinese 12 temperament,1818103,hye,https://archive.org/details/china-12-temp
1,archive_org,part02ok,1489113,hye,https://archive.org/details/part02ok
2,archive_org,金文詁林,1478816,hye,https://archive.org/details/15_20221021
3,archive_org,A[stua]tsashunch matean hin ew nor ktakaranats : est chshgri,1463541,hye,https://archive.org/details/astuatsashunchma00zohr
4,archive_org,"Армянский вестник, 1916-1918 гг. / Հայկական տեղեկագիր",1240519,hye,https://archive.org/details/1917-10-11
5,archive_org,Armenian (1733) Mekhitar BIble (WDL-15542),1158798,hyw,https://archive.org/details/HYEAMB_DBS_HS
6,archive_org,1916 ՀՈՐԻԶՈՆ / HORIZON / ГОРИЗОНТ,1135583,hyw,https://archive.org/details/1_20220118_20220118_1146
7,archive_org,Astowatsashownch Girk Hin ew Nor Ktakaranats : Ebrayakan ew,1064786,hye,https://archive.org/details/astowatsashownch00amer
8,archive_org,"A dictionary, English and Armenian",989707,hyw,https://archive.org/details/b33520707
9,archive_org,Armenian (1917) Bible,970363,hyw,https://archive.org/details/HYEB17_DBS_HS


## 17. Source × Source Type Cross-Tab

Which sources are tagged as which content type?

In [35]:
pipeline_src_type = [
    {"$group": {
        "_id": {
            "source": "$source",
            "source_type": {"$ifNull": ["$metadata.source_type", "not_set"]},
        },
        "doc_count": {"$sum": 1},
    }},
    {"$sort": {"_id.source": 1}},
]
rows_st = list(docs.aggregate(pipeline_src_type))
flat_st = [{"source": r["_id"]["source"], "source_type": r["_id"]["source_type"],
            "doc_count": r["doc_count"]} for r in rows_st]
df_st = pd.DataFrame(flat_st)
if not df_st.empty:
    pivot_st = df_st.pivot_table(index="source", columns="source_type",
                                  values="doc_count", aggfunc="sum", fill_value=0, margins=True)
    display(pivot_st)

source_type,book,cultural,dataset,encyclopedia,library,literary,news,All
source,,,,,,,,
archive_org,1570,0,0,0,0,0,0,1570
dpla,0,0,0,0,1815,0,0,1815
eastern_armenian_news:a1plus,0,0,0,0,0,0,162,162
eastern_armenian_news:aravot,0,0,0,0,0,0,135,135
eastern_armenian_news:armenpress,0,0,0,0,0,0,67,67
eastern_armenian_news:armtimes,0,0,0,0,0,0,163,163
hamazkayin:main,0,162,0,0,0,0,0,162
hamazkayin:pakine,0,0,0,0,0,965,0,965
opus_bible_hy,0,0,646,0,0,0,0,646


## 18. Quick Data Quality Flags

Scan for suspicious patterns: non-Armenian text, extremely high/low wa_scores, missing URLs, etc.

In [36]:
quality_checks = []

# 1. Documents with very low WA score (potential EA leakage in WA-tagged docs)
ea_leakage = docs.count_documents({
    "metadata.language_code": "hyw",
    "metadata.wa_score": {"$lt": 3.0, "$exists": True},
})
quality_checks.append(("WA-tagged but wa_score < 3.0 (potential EA leakage)", ea_leakage))

# 2. Documents without any URL
no_url = docs.count_documents({"$or": [
    {"metadata.url": {"$exists": False}},
    {"metadata.url": None},
    {"metadata.url": ""},
]})
quality_checks.append(("Missing URL", no_url))

# 3. Documents without word_count
no_wc = docs.count_documents({"$or": [
    {"metadata.word_count": {"$exists": False}},
    {"metadata.word_count": None},
]})
quality_checks.append(("Missing word_count", no_wc))

# 4. Documents without language_code
no_lang = docs.count_documents({"$or": [
    {"metadata.language_code": {"$exists": False}},
    {"metadata.language_code": None},
]})
quality_checks.append(("Missing language_code", no_lang))

# 5. Documents with text shorter than 100 chars
short_text_sample = list(docs.find(
    {"$expr": {"$lt": [{"$strLenCP": {"$ifNull": ["$text", ""]}}, 100]}},
    {"source": 1, "title": 1},
).limit(5))
short_text_count = docs.count_documents(
    {"$expr": {"$lt": [{"$strLenCP": {"$ifNull": ["$text", ""]}}, 100]}}
)
quality_checks.append(("Text shorter than 100 chars", short_text_count))

# 6. Documents with no document_metrics computed
no_metrics = docs.count_documents({"$or": [
    {"metadata.document_metrics": {"$exists": False}},
    {"metadata.document_metrics": None},
]})
quality_checks.append(("Missing document_metrics", no_metrics))

df_quality = pd.DataFrame(quality_checks, columns=["check", "count"])
df_quality["pct_of_total"] = (100 * df_quality["count"] / max(total, 1)).round(1).astype(str) + "%"
display(df_quality)

if short_text_sample:
    print("\nExamples of very short docs:")
    for d in short_text_sample:
        print(f"  [{d.get('source')}] {(d.get('title') or '?')[:60]}")

,check,count,pct_of_total
0,WA-tagged but wa_score < 3.0 (potential EA leakage),0,0.0%
1,Missing URL,1011150,99.5%
2,Missing word_count,0,0.0%
3,Missing language_code,0,0.0%
4,Text shorter than 100 chars,7337,0.7%
5,Missing document_metrics,343966,33.8%



Examples of very short docs:
  [wikipedia] Ազատութեան արձան (այլ կիրառումներ)
  [wikipedia] Ուլիսէս (այլ կիրառումներ)
  [wikipedia] Ֆութպոլի Աշխարհի Ախոյեանութիւնը
  [wikipedia] Ղեւոնդ (անձնանուն)
  [wikipedia] ԱՐՀԵՍՏԱՎԱՐԺՈՒԹԻՒՆԸ եւ հայ դասական դաստիարակչութիւնը


## 18a. Missing URL Breakdown by Source

Which sources are responsible for the missing-URL documents?

In [37]:
pipeline_missing_url = [
    {"$match": {"$or": [
        {"metadata.url": {"$exists": False}},
        {"metadata.url": None},
        {"metadata.url": ""},
    ]}},
    {"$group": {"_id": "$source", "count": {"$sum": 1}}},
    {"$sort": {"count": -1}},
]
rows_mu = list(docs.aggregate(pipeline_missing_url))
df_missing_url = pd.DataFrame([{"source": r["_id"], "missing_url_count": r["count"]} for r in rows_mu])
if not df_missing_url.empty:
    df_missing_url["pct_of_missing"] = (100 * df_missing_url["missing_url_count"] / df_missing_url["missing_url_count"].sum()).round(1)
    display(df_missing_url)
    print(f"\nTotal missing-URL docs: {df_missing_url['missing_url_count'].sum():,}")
else:
    print("All documents have URLs.")

,source,missing_url_count,pct_of_missing
0,opus_nllb_hy,602444,59.60
1,wikipedia_ea,330346,32.70
2,opus_multiccaligned_hy,34933,3.50
3,opus_ccaligned_hy,24814,2.50
4,wikipedia,13619,1.30
5,opus_ted2020_hy,1663,0.20
6,opus_qed_hy,1546,0.20
7,opus_wikimedia_hy,866,0.10
8,opus_bible_hy,646,0.10
9,opus_opensubtitles_hy,205,0.00



Total missing-URL docs: 1,011,150


## 18b. Short Text (<100 chars) Breakdown by Source

Which sources produce the very short documents? Helps decide whether to filter or fix at the scraper level.

In [38]:
pipeline_short_src = [
    {"$match": {"$expr": {"$lt": [{"$strLenCP": {"$ifNull": ["$text", ""]}}, 100]}}},
    {"$group": {
        "_id": "$source",
        "count": {"$sum": 1},
        "avg_len": {"$avg": {"$strLenCP": {"$ifNull": ["$text", ""]}}},
    }},
    {"$sort": {"count": -1}},
]
rows_short = list(docs.aggregate(pipeline_short_src))
df_short_src = pd.DataFrame([{
    "source": r["_id"],
    "short_doc_count": r["count"],
    "avg_char_len": round(r["avg_len"], 1),
} for r in rows_short])
if not df_short_src.empty:
    df_short_src["pct_of_short"] = (100 * df_short_src["short_doc_count"] / df_short_src["short_doc_count"].sum()).round(1)
    display(df_short_src)
    print(f"\nTotal short docs (<100 chars): {df_short_src['short_doc_count'].sum():,}")

    # Show a few examples per top source
    print("\nSample short docs per top source:")
    for src in df_short_src["source"].head(3):
        examples = list(docs.find(
            {"source": src, "$expr": {"$lt": [{"$strLenCP": {"$ifNull": ["$text", ""]}}, 100]}},
            {"title": 1, "text": 1},
        ).limit(3))
        print(f"\n  [{src}]")
        for ex in examples:
            title = (ex.get("title") or "?")[:50]
            text = (ex.get("text") or "")[:80].replace("\n", " ")
            print(f"    title: {title}")
            print(f"    text:  {text}")
else:
    print("No documents with text < 100 chars.")

,source,short_doc_count,avg_char_len,pct_of_short
0,wikipedia_ea,6268,59.60,85.40
1,dpla,975,56.20,13.30
2,wikipedia,59,63.70,0.80
3,rss_news:Google News - Armenia,33,78.10,0.40
4,rss_news:Hetq,2,54.50,0.00



Total short docs (<100 chars): 7,337

Sample short docs per top source:

  [wikipedia_ea]
    title: Թարգմանական գրականություն
    text:  ՎԵՐԱՀՂՈՒՄ Նոր ժամանակաշրջանների թարգմանությունների ցանկ հայերեն
    title: .dk
    text:  .dk, Դանիայի վերին մակարդակի ազգային դոմենն է։  Արտաքին հղումներ dk
    title: .tz
    text:  .tz Թանզանիայի վերին մակարդակի ազգային դոմենն է։  Արտաքին հղումներ tz  sv:Toppdo

  [dpla]
    title: Hayapatum : Patmichʻkʻ ew Patmutʻiwnk Hayotsʻ
    text:  
    title: Hayastaneayts' Aṛak'elakan S[urb] Ekeghets'woy pa
    text:  "Artatpats Ararat amsagrits'". Romanized record. Includes bibliographical refere
    title: Erusaghēm azateal
    text:  Romanized record.

  [wikipedia]
    title: Ազատութեան արձան (այլ կիրառումներ)
    text:  Ազատութեան արձան, Նիւ Եորք, ԱՄՆ Ազատութեան Արձան (Թիֆլիս)
    title: Ուլիսէս (այլ կիրառումներ)
    text:  5254 Ուլիսէս՝ Եուպիտերի տրոյցիներու աստեղնեակ «Ուլիսէս»՝ Ճէյմս Ճոյսի ճանչուած վէ
    title: Ֆութպոլի Աշխարհի Ախոյեանութիւնը
  